# LOKLOK qualitative analysis with human gold (Colab T4)

This notebook uses the majority labels from three human coders and reruns a local LLM against that reference. It needs no upload and no API key.

**Conditions**

1. `zero_shot`: Qwen codes all 193 human-coded segments without seeing any human labels.
2. `gold_few_shot`: 5-fold out-of-fold prompting. For each fold, examples come only from the other four folds; every human-coded segment is predicted exactly once.
3. `gold_applied`: the model uses all human labels as examples and codes only the segments without human labels. This condition is descriptive and is never reported as validation.

**Why the split matters**

Giving the model the same labels it is scored against is leakage. The 5-fold condition prevents that. The zero-shot condition is the clean baseline. The applied condition produces descriptive machine labels for the unlabelled remainder, not a validated result.

**Before running**

- Runtime > Change runtime type > **T4 GPU**.
- Run the install cell, restart the session once, then continue from the settings cell.
- Do not interpret agreement with Claude as human validation. Only the human gold columns are the reference here.


In [ ]:
# 1. GPU check and installs. Restart the session after this cell finishes.
!nvidia-smi
!pip -q install vllm rank_bm25 rapidfuzz pyyaml scikit-learn pandas sentence-transformers


In [ ]:
# 2. Settings. Edit only this cell.
MODEL_ID      = "Qwen/Qwen2.5-7B-Instruct-AWQ"
MODEL_TAG     = "qwen25_7b_awq"
N_FOLDS       = 5          # out-of-fold gold few-shot
N_EXEMPLARS   = 1          # retrieved human examples per target (keep prompt under 6144)
BATCH         = 8
MAX_MODEL_LEN = 6144
SEED          = 20261107
USE_HF_FALLBACK = False    # set True only if vLLM cannot start on the runtime
RUN_ZERO_SHOT   = True
RUN_GOLD_FEWSHOT = True
RUN_GOLD_APPLIED = True


In [ ]:
# 3. Unpack the embedded human-gold bundle. No upload.
import base64, os, sys, zipfile
from pathlib import Path

BUNDLE_B64 = "..."
work = Path('/content/qual_gold_work'); work.mkdir(parents=True, exist_ok=True)
zpath = work / 'qual_gold_bundle.zip'
zpath.write_bytes(base64.b64decode(BUNDLE_B64))
with zipfile.ZipFile(zpath) as z:
    z.extractall(work)
sys.path.insert(0, str(work))
print('extracted:', sorted(p.name for p in work.iterdir() if p.is_file()))


In [ ]:
# 4. Build units, verify the three-coder gold, and create the 5 folds.
import hashlib, pandas as pd, numpy as np
import qual_core as q
import qual_gold as g
g.hashlib = hashlib
from claude_labels_v1 import LABELS, FGD_PARTICIPANT, FGD_UNCLEAR

base = Path('/content/qual_gold_work')
cb = q.load_codebook(base / 'codebook_v1.yaml')
CODE_IDS = [c['id'] for c in cb['codes']]
units = q.build_units(base / 'FGD_transcription.csv',
                      base / 'selfreport_transcript.json',
                      FGD_PARTICIPANT, FGD_UNCLEAR)
UB = {r.unit_id: r._asdict() for r in units.itertuples()}

gold, gold_rows = g.load_gold(base / 'human_gold_majority.csv', units, CODE_IDS)
coders = g.load_coder_labels(base / 'human_coder_labels.csv', CODE_IDS)
pair = g.pairwise_coder_kappa(coders, CODE_IDS)
maj = g.majority_gold_from_coders(coders, CODE_IDS)
gold_aligned = gold.set_index('unit_id').loc[maj.index, CODE_IDS]
assert (maj.values == gold_aligned.values).all(), 'majority labels do not match gold file'

folds = g.make_kfold(gold, k=N_FOLDS, seed=SEED)
folds.to_csv(base / 'human_gold_5fold.csv', index=False)
pair.to_csv(base / 'human_coder_pairwise_kappa.csv', index=False)
pos = gold[CODE_IDS].sum().sort_values(ascending=False)
pos.to_csv(base / 'human_gold_positive_counts.csv', header=['n_positive'])

print('gold units:', len(gold), '| coders:', coders.coder.nunique(), '| codes:', len(CODE_IDS))
print('\nPairwise human kappa:')
print(pair.round(3).to_string(index=False))
print('\nFold sizes:', folds.fold.value_counts().sort_index().to_dict())
print('\nPositive reference counts by code:')
print(pos.to_string())


In [ ]:
# 5. Load the local LLM. vLLM first, Hugging Face 4-bit fallback.
import gc, torch
SCHEMA = q.json_schema(cb)
generate_fn = None

if not USE_HF_FALLBACK:
    from vllm import LLM, SamplingParams
    kw = dict(model=MODEL_ID, dtype='half', max_model_len=MAX_MODEL_LEN,
              gpu_memory_utilization=0.85, enforce_eager=True, seed=SEED)
    if 'AWQ' in MODEL_ID.upper():
        kw['quantization'] = 'awq'
    llm = LLM(**kw)
    def make_sp(temp=0.0, seed=SEED):
        p = dict(temperature=temp, max_tokens=400, seed=seed)
        try:
            from vllm.sampling_params import GuidedDecodingParams
            p['guided_decoding'] = GuidedDecodingParams(json=SCHEMA)
        except Exception:
            try:
                from vllm.sampling_params import StructuredOutputsParams
                p['structured_outputs'] = StructuredOutputsParams(json=SCHEMA)
            except Exception:
                pass
        try:
            return SamplingParams(**p)
        except TypeError:
            p.pop('guided_decoding', None); p.pop('structured_outputs', None)
            return SamplingParams(**p)
    SP = make_sp()
    def generate_fn(batch_msgs):
        outs = llm.chat(batch_msgs, SP, use_tqdm=False)
        return [o.outputs[0].text for o in outs]
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    HF_ID = MODEL_ID.replace('-AWQ', '')
    tok = AutoTokenizer.from_pretrained(HF_ID)
    model = AutoModelForCausalLM.from_pretrained(
        HF_ID, device_map='auto', torch_dtype=torch.float16,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4'))
    def generate_fn(batch_msgs):
        outs = []
        for m in batch_msgs:
            ids = tok.apply_chat_template(m, add_generation_prompt=True, return_tensors='pt').to(model.device)
            out = model.generate(ids, max_new_tokens=350, do_sample=False)
            outs.append(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))
        return outs

smoke = units[units.unit_id.isin(gold.unit_id[:2])].to_dict('records')
print(generate_fn([g.build_messages(u, cb) for u in smoke]))


In [ ]:
# 6. Zero-shot baseline on all human-coded segments.
out_dir = Path('/content/qual_gold_out'); out_dir.mkdir(parents=True, exist_ok=True)
zero_rows = []
if RUN_ZERO_SHOT:
    zero_rows = g.run_units(units, cb, generate_fn, gold.unit_id.tolist(),
                            out_dir / f'{MODEL_TAG}_zero_shot.jsonl',
                            pool=None, condition='zero_shot', batch=BATCH)
print('zero-shot rows:', len(zero_rows))


In [ ]:
# 7. Gold few-shot: 5-fold out-of-fold. This is the leakage-safe LLM validation.
few_rows, folds = ([], folds) if not RUN_GOLD_FEWSHOT else g.run_kfold(
    gold, gold_rows, units, cb, generate_fn, k=N_FOLDS, seed=SEED,
    path=str(out_dir / f'{MODEL_TAG}_gold_fewshot.jsonl'),
    n_exemplars=N_EXEMPLARS, batch=BATCH)
print('few-shot rows:', len(few_rows), '| expected:', len(gold))


In [ ]:
# 8. Gold applied: all human labels as examples, only unlabelled units coded.
# Descriptive only. Never report this as validation.
applied_rows = []
if RUN_GOLD_APPLIED:
    all_pool = g.GoldExemplarPool(units, gold_rows, gold.unit_id.tolist(), k=N_EXEMPLARS)
    unlabelled = units[
        (~units.unit_id.isin(set(gold.unit_id))) & units.role_ref.isin(q.EVAL_ROLES)
    ].unit_id.tolist()
    applied_rows = g.run_units(units, cb, generate_fn, unlabelled,
                               out_dir / f'{MODEL_TAG}_gold_applied.jsonl',
                               pool=all_pool, condition='gold_applied', batch=BATCH)
print('applied rows:', len(applied_rows))


In [ ]:
# 9. Evaluate zero-shot and gold few-shot against the full human gold.
# Claude's static first pass is included for comparison only; it is not re-run.
import pandas as pd
from IPython.display import display

eval_ids = gold.unit_id.tolist()
ref_claude = q.labels_to_rows(LABELS, units)
summaries, tables = [], {}
for name, rows in [('qwen_zero_shot', zero_rows),
                   ('qwen_gold_fewshot', few_rows),
                   ('claude_static', ref_claude)]:
    kept, tab, s = g.evaluate(rows, gold_rows, units, cb, eval_ids=eval_ids, label=name)
    tab.to_csv(out_dir / f'human_vs_{name}.csv', index=False)
    tables[name] = tab
    summaries.append(s)
    print('\n==', name, '==')
    print('micro-F1:', round(s['micro_f1'], 3),
          '| exact fabrication:', round(s['fabrication_rate_exact'], 3),
          '| role accuracy:', round(s['role_accuracy'], 3))
    display(tab[tab.n_ref > 0][['code_id', 'n_ref', 'n_test', 'kappa',
                               'kappa_lo', 'kappa_hi', 'precision', 'recall',
                               'decision']].round(3))

S = pd.DataFrame(summaries)
S.to_csv(out_dir / 'summary_human_gold.csv', index=False)
print('\nSummary:')
display(S[['label', 'micro_f1', 'fabrication_rate_exact', 'role_accuracy',
           'parse_failure_rate', 'insufficient_evidence_rate', 'n_eval_units']].round(3))

# Code-level agreement between the two Qwen conditions.
a = tables['qwen_zero_shot'].set_index('code_id')
b = tables['qwen_gold_fewshot'].set_index('code_id')
comp = a[['n_ref', 'kappa']].rename(columns={'kappa': 'kappa_zero'}).join(
    b[['n_test', 'kappa']].rename(columns={'kappa': 'kappa_gold'}))
comp['delta_kappa'] = comp.kappa_gold - comp.kappa_zero
comp.to_csv(out_dir / 'condition_comparison.csv')
display(comp.round(3))


In [ ]:
# 10. Descriptive applied coding for the unlabelled remainder.
prev = g.prevalence(applied_rows, units, set(gold.unit_id), label='gold_applied')
prev.to_csv(out_dir / 'applied_prevalence.csv', index=False)
assign = g.flatten_assignments(applied_rows, units, label='gold_applied')
assign.to_csv(out_dir / 'applied_assignments.csv', index=False)
print('Applied rows:', len(applied_rows), '| assignments:', len(assign))
display(prev)
print('\nPer-source predictions:')
if len(assign):
    print(assign.groupby(['source', 'code_id']).size().unstack(fill_value=0).to_string())


In [ ]:
# 11. Package results for download.
import shutil
from google.colab import files
shutil.make_archive('/content/loklok_qual_gold_results', 'zip', out_dir)
try:
    files.download('/content/loklok_qual_gold_results.zip')
except Exception:
    print('Download /content/loklok_qual_gold_results.zip from the file browser.')


## How to read the results

- **Human gold**: majority of the three coders, 193 FGD/self-report segments, 25 binary codes. Pairwise kappa is printed in cell 4.
- **`qwen_zero_shot`**: baseline with no human examples. This is the clean comparison.
- **`qwen_gold_fewshot`**: 5-fold out-of-fold. Every gold segment is scored once, and examples come only from the other folds. This is the leakage-safe human-gold LLM result.
- **`claude_static`**: the earlier first-pass labels, evaluated on the same gold. It was not regenerated in this notebook.
- **`gold_applied`**: all human labels are examples, and only unlabelled units are coded. These are descriptive machine labels, not validation.

Do not describe the applied condition as human-validated. Do not pool segment counts into student prevalence. Report exact-quote fabrication for every condition, and report the pre-specified per-code decision rather than only micro-F1.
